In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# SE 2025 - Lab 3 : Evaluation

In this lab we will revise the evaluation process of IR systems.

In [ ]:
%%capture
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

In [ ]:
import numpy as np
import pandas as pd
import tqdm
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

## The dataset

First, we will introduce the dataset. In our labs, we will be using a subset of the small version of [WikIR](https://www.aclweb.org/anthology/2020.lrec-1.237.pdf) dataset for English.

Download the following files (available also on Absalon under folder `lab`) in a folder called `data/`:
- [`lab_docs.csv`](https://absalon.instructure.com/files/7103632/download?download_frd=1): CSV file of document number and document text
- [`lab_topics.csv`](https://absalon.instructure.com/files/7103631/download?download_frd=1): CSV file of query id and query text
- [`lab_qrels.csv`](https://absalon.instructure.com/files/7103630/download?download_frd=1): CSV file of annotations with schema `qid, docno, label, iteration`

In [ ]:
docs = pd.read_csv('lab_docs.csv', dtype={'docno': str, 'text': str})
topics = pd.read_csv('lab_topics.csv', dtype={'qid': str})
qrels = pd.read_csv('lab_qrels.csv', dtype={'qid': str, 'docno': str, 'label': int})

## Evaluation

In [ ]:
%%capture
!pip install python-terrier

In [ ]:
%env JAVA_HOME=/root/.sdkman/candidates/java/current
import pyterrier as pt
if not pt.started():
    pt.init()

In [ ]:
indexer = pt.IterDictIndexer("./indexes/pt_index_default", overwrite=True, blocks=True)
index_ref = indexer.index(docs.to_dict(orient='records'))
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

Evaluating an IR model involves the followings:
- We have a number of queries/topics.
- We have a ranking model.
- We have access to assessed runs (aka ground-truth relevance, aka qrels).
- Loop over queries:
    - Using the ranking model, we rank documents for a given query.
    - We use the qrels for the given query to produce a score for the model's ranking.
- Average the score over the entire set of queries.

Let's take a look at the qrels:

In [ ]:
np.unique(qrels['label'], return_counts=True)

For this dataset, 2 means high-relevant, 1 relevant and any missing qid-docno pair means not-relevant. We can check how many relevant document are there per query:

In [ ]:
qrels.groupby(['qid', 'label'], as_index=False).agg({'docno': 'count'}).sort_values(['qid', 'label'])

It makes sense to compute a ground truth query-document matrix:

In [ ]:
unique_qids = qrels['qid'].unique()
unique_docno = docs['docno'].unique()

# qids and docno need to map to rows and columns, respectively,
# so we need to remap them
qid2row = dict(zip(unique_qids, range(len(unique_qids))))
docno2col = dict(zip(unique_docno, range(len(unique_docno))))

query_doc_gt = np.zeros((len(unique_qids), len(unique_docno)))
query_doc_gt.shape

In [ ]:
# Lets fill it up
for _, row in qrels.iterrows():
    rowid = qid2row[row['qid']]
    colid = docno2col[row['docno']]
    query_doc_gt[rowid, colid] = row['label']
query_doc_gt

For a first taste on IR evaluation, we would like to measure performance of a BM25 ranker on the lab dataset.

In [ ]:
BM25 = pt.terrier.Retriever(index, wmodel="BM25")

Remember that the above creates a model with the following default hyperparameters:
- k1 = 1.2
- b = 0.75

We would like to track the following metrics:
- Precision@10
- Recall@10
- MAP@10

In [ ]:
K = 10
precision = []
recall = []
map = []

for _, row in topics.iterrows():
    rowid = row['qid']
    query = row['query']
    ground_truth = query_doc_gt[qid2row[rowid]]
    RB = np.sum(ground_truth > 0) # Relevant documents (True positives + False negatives)

    results = BM25.search(query).head(K)   # (qid, docid, docno, rank, score, query) [D2, D5, ..., D88]
    results2col = [docno2col[docno] for docno in results['docno']]  # Documentd IDs
    results_relevance = ground_truth[results2col]  # [2, 1, 1, ..., 2]

    # All metrics require a binary relevance
    results_relevance_binary = results_relevance > 0  # [1, 1, 0, 1,..., 1]

    #TODO: compute the metrics
    precision.append(np.mean(results_relevance_binary))
    recall.append(np.sum(results_relevance_binary) / RB)

    prec_K = results_relevance_binary * np.cumsum(results_relevance_binary) / (1 + np.arange(K))

    map.append(np.sum )
    #EndTODO

print(f"Precision@{K}: {round(np.mean(precision), 4)}")
print(f"Recall@{K}: {round(np.mean(recall), 4)}")
print(f"MAP@{K}: {round(np.mean(map), 4)}")

#### Q: what happens with Recall@10 if a certain query has only 5 true relevant documents and a model ranks all 5 at the top of the list with 10 documents?

We can perform the same task by using directly PyTerrier. It provides an interface for evaluating the performance of IR systems through the `Experiment` abstraction. Behind the scenes, `pt.Experiment` uses the `pytrec_eval` library:

In [ ]:
pt.Experiment(
    retr_systems=[BM25],
    names=['BM25'],
    topics=topics,
    qrels=qrels,
    eval_metrics=[f'P_{K}', f'recall_{K}', f'map_cut_{K}'],
    round=4
)